# Seq Len Hidden State Debug


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from pfns.experiments.model_benchmarks.model_registry import get_models_from_names
from pfns.utils import get_default_device

cwd = Path.cwd()
candidate_dirs = [cwd, cwd / 'PFNs' / 'notebooks']
notebook_dir = next((p for p in candidate_dirs if (p / 'seq_len_debug_utils.py').exists()), None)
if notebook_dir is None:
    raise FileNotFoundError('Could not locate seq_len_debug_utils.py')
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

from seq_len_debug_utils import (
    plot_hidden_state_metric,
    plot_recurrent_metric_per_head,
    run_hidden_state_tracking,
    summarize_hidden_state_by_seqlen,
)

plt.rcParams['figure.dpi'] = 400


## Config


In [ ]:
EXPERIMENT = {
    'name': 'seq_len_hidden_state_debug',
    'num_classes': 5,
    'num_features': 10,
    'num_test_samples': 100,
    'num_repetitions': 20,
    'data_generation_seed': 42,
    'seqlen_list': [250, 500, 1_000, 2_000, 4_000, 8_000, 16_000, 32_000],
}

MODEL_NAMES = [
    'DeltaNet_Comb_MT',
]

# Optional name filters. Example: ['cache', 'kv']
TENSOR_NAME_PATTERNS = None

device = str(get_default_device())
models_to_compare = get_models_from_names(MODEL_NAMES)

print(f'Device: {device}')
print(f'Models: {list(models_to_compare)}')
print(f'Repetitions: {EXPERIMENT["num_repetitions"]}')
print(f'Seq lens: {EXPERIMENT["seqlen_list"]}')



## Run Hidden-State Tracking


In [ ]:
hidden_state_df = run_hidden_state_tracking(
    experiment=EXPERIMENT,
    models_to_compare=models_to_compare,
    device=device,
    tensor_name_patterns=TENSOR_NAME_PATTERNS,
)

print('hidden_state_df rows:', len(hidden_state_df))
hidden_state_df.head()



## Summaries and Plots


In [ ]:
summary_df = summarize_hidden_state_by_seqlen(hidden_state_df)
tensor_summary_df = summary_df[summary_df['state_scope'] == 'tensor'] if 'state_scope' in summary_df.columns else summary_df
recurrent_summary_df = summary_df[summary_df['state_scope'] == 'recurrent_state_head'] if 'state_scope' in summary_df.columns else pd.DataFrame()


### Metric Guide
- `l2_norm_mean`: average vector norm of a tracked tensor.
- `abs_max_mean`: average largest absolute value in a tracked tensor.
- `mean_mean`: average element mean of a tracked tensor.
- `std_mean`: average element standard deviation of a tracked tensor.
- `finite_frac_mean`: fraction of finite entries (`1.0` means no NaN/Inf).
- `fro_norm_mean`: per-head Frobenius norm of `recurrent_state` matrices.
- `spectral_norm_mean`: per-head spectral norm (largest singular value) of `recurrent_state` matrices.
- `cond_proxy_mean`: per-head condition proxy `sigma_max / max(sigma_min, 1e-12)` for `recurrent_state` matrices.


In [ ]:
plot_df = tensor_summary_df if not tensor_summary_df.empty else summary_df
all_tensors = sorted(plot_df['tensor_name'].astype(str).unique().tolist())
print(f'Plotting generic hidden-state tensors: {len(all_tensors)}')
for name in all_tensors:
    print(' ', name)
if not recurrent_summary_df.empty:
    print(f'Recurrent-state head rows available: {len(recurrent_summary_df)}')


In [ ]:
summary_metrics = [
    ('l2_norm_mean', 'Hidden-state L2 norm vs sequence length'),
    ('abs_max_mean', 'Hidden-state abs max vs sequence length'),
    ('mean_mean', 'Hidden-state mean vs sequence length'),
    ('std_mean', 'Hidden-state std vs sequence length'),
    ('finite_frac_mean', 'Hidden-state finite fraction vs sequence length'),
]

for metric, title in summary_metrics:
    if metric not in plot_df.columns:
        continue
    plot_hidden_state_metric(
        plot_df,
        metric=metric,
        title=title,
        tensor_names=all_tensors,
    )

recurrent_metrics = [
    ('fro_norm_mean', 'Recurrent-state Frobenius norm vs sequence length'),
    ('spectral_norm_mean', 'Recurrent-state spectral norm vs sequence length'),
    ('cond_proxy_mean', 'Recurrent-state condition proxy vs sequence length'),
]

for metric, title in recurrent_metrics:
    if metric not in recurrent_summary_df.columns:
        continue
    plot_recurrent_metric_per_head(
        recurrent_summary_df,
        metric=metric,
        title_prefix=title,
    )


## Save Outputs


In [ ]:
SAVE_DIR = Path('exp_outputs/seq_len/hidden_state_debug')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

hidden_state_df.to_csv(SAVE_DIR / 'hidden_state.csv', index=False)
summary_df.to_csv(SAVE_DIR / 'hidden_state_summary_by_seqlen.csv', index=False)

print(f'Saved outputs to: {SAVE_DIR.resolve()}')
